# Quantum Field Theory Analysis of Neural Networks

This notebook demonstrates the new QFT module in WeightWatcher, which implements theoretical foundations from quantum field theory and renormalization group theory to analyze neural network weight matrices.

## Theoretical Background

The QFT module is based on the theory that neural networks approach a critical point during training, which can be described as a kind of fractal where the free energy satisfies scale invariance, according to Wilson's exact renormalization group theory.

Key concepts implemented:
1. **Critical Points**: Points where the system exhibits scale invariance
2. **Fractal Dimension**: Measure of self-similarity across scales
3. **Free Energy Landscape**: Mapping the thermodynamic properties of weight matrices
4. **Phase Transitions**: Detecting significant changes in network behavior
5. **Universality Classes**: Classifying networks based on critical exponents

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import weightwatcher as ww
from weightwatcher import RGAnalyzer

# For demonstration with real models
try:
    import tensorflow as tf
    import keras
    HAS_KERAS = True
except ImportError:
    HAS_KERAS = False
    print("Keras not available. Some examples will be skipped.")

## 1. Analyzing Synthetic Weight Matrices

Let's start by analyzing synthetic weight matrices with different properties.

In [ ]:
# Create an RG Analyzer
rg_analyzer = RGAnalyzer(temperature=1.0)

# Generate synthetic weight matrices
np.random.seed(42)

# 1. Random Gaussian matrix (far from critical)
W_random = np.random.normal(0, 1, (1000, 500))

# 2. Power-law distributed singular values (near critical)
U, _, V = np.linalg.svd(np.random.normal(0, 1, (1000, 500)), full_matrices=False)
s = np.power(np.arange(1, 501), -1)  # Power law with exponent -1
W_critical = U @ np.diag(s) @ V

# 3. Exponentially distributed singular values (ordered, far from critical)
s_exp = np.exp(-np.arange(500) / 50)
W_ordered = U @ np.diag(s_exp) @ V

In [ ]:
# Analyze the matrices
results_random = rg_analyzer.track_rg_flow(W_random, epoch=0)
results_critical = rg_analyzer.track_rg_flow(W_critical, epoch=1)
results_ordered = rg_analyzer.track_rg_flow(W_ordered, epoch=2)

# Display key metrics
metrics = ['power_law_exponent', 'scale_invariance', 'fractal_dimension', 'free_energy', 'is_critical']
matrices = {'Random': results_random, 'Critical': results_critical, 'Ordered': results_ordered}

for name, results in matrices.items():
    print(f"\n{name} Matrix:")
    for metric in metrics:
        if metric in results:
            print(f"  {metric}: {results[metric]}")

## 2. Visualizing the RG Flow

Now let's visualize the RG flow across these different matrices.

In [ ]:
# Visualize the RG flow
fig = rg_analyzer.visualize_rg_flow(figsize=(14, 12))
plt.show()

## 3. Detecting Phase Transitions

Let's detect if there's a phase transition between our synthetic matrices.

In [ ]:
# Detect phase transitions
transition1 = rg_analyzer.detect_phase_transition(W_random, W_critical)
transition2 = rg_analyzer.detect_phase_transition(W_critical, W_ordered)

print("\nPhase Transition from Random to Critical:")
print(f"  Is phase transition: {transition1['is_phase_transition']}")
print(f"  Transition type: {transition1['transition_type']}")
print(f"  Power law difference: {transition1['power_law_diff']:.4f}")

print("\nPhase Transition from Critical to Ordered:")
print(f"  Is phase transition: {transition2['is_phase_transition']}")
print(f"  Transition type: {transition2['transition_type']}")
print(f"  Power law difference: {transition2['power_law_diff']:.4f}")

## 4. Analyzing Correlation Length

Correlation length diverges at critical points, providing another measure of criticality.

In [ ]:
# Analyze correlation length
corr_random = rg_analyzer.analyze_correlation_length(W_random)
corr_critical = rg_analyzer.analyze_correlation_length(W_critical)
corr_ordered = rg_analyzer.analyze_correlation_length(W_ordered)

print("\nCorrelation Length Analysis:")
print(f"  Random Matrix: {corr_random['correlation_length']:.4f} (decay: {corr_random['correlation_decay']:.4f})")
print(f"  Critical Matrix: {corr_critical['correlation_length']:.4f} (decay: {corr_critical['correlation_decay']:.4f})")
print(f"  Ordered Matrix: {corr_ordered['correlation_length']:.4f} (decay: {corr_ordered['correlation_decay']:.4f})")

## 5. Determining Universality Classes

Let's classify our matrices into known universality classes from statistical physics.

In [ ]:
# Compute universality classes
univ_random = rg_analyzer.compute_universality_class(W_random)
univ_critical = rg_analyzer.compute_universality_class(W_critical)
univ_ordered = rg_analyzer.compute_universality_class(W_ordered)

print("\nUniversality Class Analysis:")
print(f"  Random Matrix: {univ_random['universality_class']} (confidence: {univ_random['confidence']:.2f})")
print(f"  Critical Matrix: {univ_critical['universality_class']} (confidence: {univ_critical['confidence']:.2f})")
print(f"  Ordered Matrix: {univ_ordered['universality_class']} (confidence: {univ_ordered['confidence']:.2f})")

## 6. Analyzing Real Neural Networks (if Keras is available)

If Keras is available, let's analyze a real neural network.

In [ ]:
if HAS_KERAS:
    # Load a pre-trained model or create a simple one
    model = keras.applications.VGG16(weights='imagenet', include_top=True)
    
    # Extract weight matrices from convolutional and dense layers
    weight_matrices = []
    layer_names = []
    
    for layer in model.layers:
        if hasattr(layer, 'kernel'):
            weights = layer.kernel.numpy()
            if len(weights.shape) == 4:  # Conv layer
                # Reshape to 2D matrix
                w_reshaped = weights.reshape(weights.shape[0] * weights.shape[1] * weights.shape[2], weights.shape[3])
                weight_matrices.append(w_reshaped)
            else:  # Dense layer
                weight_matrices.append(weights)
            layer_names.append(layer.name)
    
    # Analyze each weight matrix
    print("\nAnalyzing VGG16 layers:")
    for i, (W, name) in enumerate(zip(weight_matrices, layer_names)):
        results = rg_analyzer.track_rg_flow(W, epoch=i)
        print(f"\nLayer: {name}")
        print(f"  Power Law Exponent: {results['power_law_exponent']:.4f}")
        print(f"  Scale Invariance: {results['scale_invariance']:.4f}")
        print(f"  Fractal Dimension: {results['fractal_dimension']:.4f}")
        print(f"  Is Critical: {results['is_critical']}")
    
    # Visualize the RG flow across layers
    fig = rg_analyzer.visualize_rg_flow(figsize=(14, 12))
    plt.show()
else:
    print("Keras not available. Skipping real neural network analysis.")

## 7. Comparing with Traditional WeightWatcher Metrics

Let's compare our QFT-based metrics with traditional WeightWatcher metrics.

In [ ]:
# Initialize traditional WeightWatcher
watcher = ww.WeightWatcher()

# Analyze synthetic matrices with traditional metrics
details_random = watcher.analyze(W_random, layer_id=0, plot=False)
details_critical = watcher.analyze(W_critical, layer_id=1, plot=False)
details_ordered = watcher.analyze(W_ordered, layer_id=2, plot=False)

# Compare with QFT metrics
print("\nComparison of Traditional vs QFT Metrics:")
print("\nRandom Matrix:")
print(f"  Traditional - Alpha: {details_random['alpha']:.4f}, Stable Rank: {details_random['stable_rank']:.4f}")
print(f"  QFT - Power Law: {results_random['power_law_exponent']:.4f}, Fractal Dim: {results_random['fractal_dimension']:.4f}")

print("\nCritical Matrix:")
print(f"  Traditional - Alpha: {details_critical['alpha']:.4f}, Stable Rank: {details_critical['stable_rank']:.4f}")
print(f"  QFT - Power Law: {results_critical['power_law_exponent']:.4f}, Fractal Dim: {results_critical['fractal_dimension']:.4f}")

print("\nOrdered Matrix:")
print(f"  Traditional - Alpha: {details_ordered['alpha']:.4f}, Stable Rank: {details_ordered['stable_rank']:.4f}")
print(f"  QFT - Power Law: {results_ordered['power_law_exponent']:.4f}, Fractal Dim: {results_ordered['fractal_dimension']:.4f}")

## 8. Conclusion

The QFT module provides deeper theoretical insights into neural network behavior by implementing concepts from quantum field theory and renormalization group theory. Key findings:

1. **Critical Points**: Neural networks tend to perform best when their weight matrices approach critical points
2. **Scale Invariance**: Critical weight matrices exhibit scale invariance properties
3. **Fractal Structure**: The eigenvalue/singular value distributions form fractal-like structures
4. **Phase Transitions**: Significant changes in training can be detected as phase transitions
5. **Universality Classes**: Neural networks can be classified into known universality classes from statistical physics

These insights can help guide architecture design, initialization strategies, and training procedures to optimize neural network performance.